# APIs: Building a DataFrame & Saving Data
### OIM 7502 — API Series, Part 2 of 4

In [ ]:
import json
import requests
import pandas as pd

endpoint = "https://dummyjson.com/posts"
req = requests.get(endpoint).text
data = json.loads(req)

### From JSON to DataFrame
A list of similarly-shaped dicts converts directly into a DataFrame — one row per dict, one column per key.

In [ ]:
df = pd.DataFrame(data['posts'])
df.head()

### Saving the data — and a bug worth knowing about
Here's a mistake that's easy to make and easy to miss: writing an *already-JSON-encoded string* through `json.dump` a second time.

In [ ]:
# DON'T do this -- req is already a JSON string, and json.dump() encodes it AGAIN
with open('data/test_bad.txt', 'w') as f:
    json.dump(req, f)

with open('data/test_bad.txt') as f:
    print(f.read()[:200])

See the problem? The file now contains a JSON string *of* a JSON string — every quote inside is escaped (`\"`), and if you read this file back with `json.loads`, you'd get a plain string back, not the dict/list you actually want. You'd have to call `json.loads` on it **twice** to undo the damage.

There are two correct ways to save this, depending on what you're starting from:

In [ ]:
# Option A: you already have TEXT (like req) -- just write it directly, no json.dump needed
with open('data/test_text.txt', 'w') as f:
    f.write(req)

In [ ]:
# Option B: you have a Python OBJECT (like data, or df) -- json.dump is correct here,
# because it's encoding an object for the first time, not re-encoding text
with open('data/test_object.json', 'w') as f:
    json.dump(data, f)

**Rule of thumb:** `json.dump`/`json.dumps` encode a Python object into JSON text. If what you're holding is already text (like the raw `.text` from a `requests` response), just write it — don't hand it to `json.dump` again.